In [47]:
print(ord('a'))
# character -> ASCII value
# This function returns the ASCII value of the character given as argument
print(bytes([97]))
# ASCII value -> bytes
# In the ASCII table, the value 97 represents the a character
print(chr(256))

97
b'a'
Ā


In [37]:
list(" go".encode("utf-8"))

[32, 103, 111]

In [740]:
import regex as re

pattern = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
compiled_pattern = re.compile(pattern=pattern)

text = "你好。"
text_chunks = re.findall(compiled_pattern, text)
ids = [list(ch.encode("utf-8")) for ch in text_chunks]
vocab = {idx: bytes([idx]) for idx in range(256)}

print(f"text:\n{text}")
print(f"text_chunks:\n{text_chunks}")
print(f"intial ids:\n{ids}")
print(f"initial vocab:\n{vocab}")

merges: dict[tuple[int, int], int] = {}

def get_stats(ids, counts=None):
  """
  Given a list of integers, return a dictionary of counts of consecutive pairs
  Example: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
  Optionally allows to update an existing dictionary of counts
  """
  counts = {} if counts is None else counts
  for pair in zip(ids, ids[1:]): # iterate consecutive elements
    counts[pair] = counts.get(pair, 0) + 1
  return counts

def merge(ids, pair, idx):
  """
  In the list of integers (ids), replace all consecutive occurrences
  of pair with the new integer token idx
  Example: ids=[1, 2, 3, 1, 2], pair=(1, 2), idx=4 -> [4, 3, 4]
  """
  newids = []
  i = 0
  while i < len(ids):
    # if not at the very last position AND the pair matches, replace it
    if ids[i] == pair[0] and i < len(ids) - 1 and ids[i+1] == pair[1]:
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids

vocab_size = 256
num_merges = vocab_size - 256
for i in range(num_merges):
  stats = {}
  # count the number of times every consecutive pair appears
  for chunk_ids in ids:
    # passing in stats will update it in place, adding up counts
    get_stats(chunk_ids, stats)
  print(f"i: {i}, stats: { {b''.join([vocab[p0], vocab[p1]]).decode('utf-8'): frequency for (p0, p1), frequency in stats.items()} }")
  pair = max(stats, key=stats.get)
  idx = 256 + i
  ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]
  merges[pair] = idx
  vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

print(f"merges:\n{merges}")
print(f"merges in charcaters:\n{ {vocab[idx].decode('utf-8'): idx for (_, _), idx in merges.items()} }")
# print(f"'{bytes([32, 103]).decode('utf-8')}', '{bytes([32, 103, 111]).decode('utf-8')}', {bytes([32, 97])}, {bytes([32, 97, 98])}")
# ' g', ' go', ' a', ' ab'
print(f"final vocab:\n{vocab}")
print(f"final ids:\n{ids}")

text:
你好。
text_chunks:
['你好', '。']
intial ids:
[[228, 189, 160, 229, 165, 189], [227, 128, 130]]
initial vocab:
{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P

In [744]:
import sys
import os
root_path = os.path.abspath('..')
if root_path not in sys.path:
  sys.path.append(root_path)

from importlib import reload
from src import base, regex
reload(base)
reload(regex)
from src.regex import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.train("deep learning engineer", 260)
print([{tokenizer.vocab[idx].decode('utf-8'): idx} for _, idx in tokenizer.merges.items()])

[{'ee': 256}, {'in': 257}, {'dee': 258}, {'deep': 259}]


# check "dog. dog! dog?" problem

Both suffers from the tokens of "dog" and "_dog". In `minbpe`, this happens if we try to maximize the vocab_size. In `SentencePiece`, this happens midway (before the vocab max size).

The consequence of using SentencePiece is that the LLM model will be more prone to the "SolidGoldMagikarp" problem (i.e. undertrained token) because from training set "dog. dog! dog?", we have extra token "og", "_d", "_do".

1. minbpe (vocab size 256 fallback + 3 bpe tokens): [{'do': 256}, {'dog': 257}, {' dog': 258}]
2. SentencePiece (vocab size 256 fallback + 3 special tokens + 7 single tokens + 3 bpe tokens): [['do', 259], ['dog', 260], ['▁dog', 261], ['d', 262], ['g', 263], ['o', 264], ['▁', 265], ['!', 266], ['.', 267], ['?', 268]]
3. SentencePiece (vocab size 256 fallback + 3 special tokens + 7 single tokens + 3 bpe tokens + 3 no idea): [['do', 259], ['dog', 260], ['▁dog', 261], ['og', 262], ['▁d', 263], ['▁do', 264], ['d', 265], ['g', 266], ['o', 267], ['▁', 268], ['!', 269], ['.', 270], ['?', 271]]

In [606]:
import sys
import os
root_path = os.path.abspath('..')
if root_path not in sys.path:
  sys.path.append(root_path)

from importlib import reload
from src import base, regex
reload(base)
reload(regex)
from src.regex import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.train("dog. dog! dog?", 256 + 3)
print([{tokenizer.vocab[idx].decode('utf-8'): idx} for _, idx in tokenizer.merges.items()])

[{'do': 256}, {'dog': 257}, {' dog': 258}]


In [616]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
  sentence_iterator=iter(["dog. dog! dog?"]),
  model_prefix="output/spm_model",
  model_type="bpe",
  vocab_size=256 + 3 + 7 + 3, # 256 fallback + 3 special tokens + 7 single tokens + 3 bpe tokens + 3 no idea
  byte_fallback=True
)

sp = spm.SentencePieceProcessor()
sp.Load(model_file="output/spm_model.model")
vocab = [[sp.IdToPiece(idx), idx] for idx in range(sp.GetPieceSize())]
print(vocab) # check the vocab order, it's changed.
print(f"dog.: {sp.EncodeAsPieces('dog.')}")
print(f"dog.: {sp.Encode('dog.dog')}")
print(f"dog!: {sp.EncodeAsPieces('dog!')}")
print(f"dog!: {sp.Encode('dog!')}")
print(f"dog?: {sp.EncodeAsPieces('dog?')}")
print(f"dog?: {sp.Encode('dog?')}")

[['<unk>', 0], ['<s>', 1], ['</s>', 2], ['<0x00>', 3], ['<0x01>', 4], ['<0x02>', 5], ['<0x03>', 6], ['<0x04>', 7], ['<0x05>', 8], ['<0x06>', 9], ['<0x07>', 10], ['<0x08>', 11], ['<0x09>', 12], ['<0x0A>', 13], ['<0x0B>', 14], ['<0x0C>', 15], ['<0x0D>', 16], ['<0x0E>', 17], ['<0x0F>', 18], ['<0x10>', 19], ['<0x11>', 20], ['<0x12>', 21], ['<0x13>', 22], ['<0x14>', 23], ['<0x15>', 24], ['<0x16>', 25], ['<0x17>', 26], ['<0x18>', 27], ['<0x19>', 28], ['<0x1A>', 29], ['<0x1B>', 30], ['<0x1C>', 31], ['<0x1D>', 32], ['<0x1E>', 33], ['<0x1F>', 34], ['<0x20>', 35], ['<0x21>', 36], ['<0x22>', 37], ['<0x23>', 38], ['<0x24>', 39], ['<0x25>', 40], ['<0x26>', 41], ['<0x27>', 42], ['<0x28>', 43], ['<0x29>', 44], ['<0x2A>', 45], ['<0x2B>', 46], ['<0x2C>', 47], ['<0x2D>', 48], ['<0x2E>', 49], ['<0x2F>', 50], ['<0x30>', 51], ['<0x31>', 52], ['<0x32>', 53], ['<0x33>', 54], ['<0x34>', 55], ['<0x35>', 56], ['<0x36>', 57], ['<0x37>', 58], ['<0x38>', 59], ['<0x39>', 60], ['<0x3A>', 61], ['<0x3B>', 62], ['<0x3C

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input_format: 
  model_prefix: output/spm_model
  model_type: BPE
  vocab_size: 269
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noi

ce.cc(425) LOG(INFO) Adding meta_piece: <0xCA>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xCB>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xCC>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xCD>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xCE>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xCF>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD0>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD1>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD2>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD3>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD4>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD5>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD6>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD7>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD8>
trainer_interface.cc(425) LOG(INFO) Adding meta_piece: <0xD9>
trainer_interface.cc(42

In [875]:
import sys
import os
root_path = os.path.abspath('..')
if root_path not in sys.path:
  sys.path.append(root_path)

from importlib import reload
from src import base, regex
reload(base)
reload(regex)
from src.regex import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.train("I've we've you've they've", 256 + 11)
print([{tokenizer.vocab[idx].decode('utf-8'): idx} for _, idx in tokenizer.merges.items()])
print(f"I've we've you've they've:", [tokenizer.vocab[idx].decode('utf-8') for idx in tokenizer.encode("I've we've you've they've")])
print(f"I've we've you've they've:", tokenizer.decode(tokenizer.encode("I've we've you've they've")))

[{"'v": 256}, {"'ve": 257}, {' w': 258}, {' we': 259}, {' y': 260}, {' yo': 261}, {' you': 262}, {' t': 263}, {' th': 264}, {' the': 265}, {' they': 266}]
I've we've you've they've: ['I', "'ve", ' we', "'ve", ' you', "'ve", ' they', "'ve"]
I've we've you've they've: I've we've you've they've


In [874]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
  sentence_iterator=iter(["I've we've you've they've"]),
  model_prefix="output/spm_model",
  model_type="bpe",
  vocab_size=256 + 3 + 11 + 18, # 256 fallback + 3 special tokens + 11 single tokens + 18 bpe tokens
  byte_fallback=True
)

sp = spm.SentencePieceProcessor()
sp.Load(model_file="output/spm_model.model")
vocab = [[sp.IdToPiece(idx), idx] for idx in range(sp.GetPieceSize())]
print(vocab) # check the vocab order, it's changed.
print(len(set("I've we've you've they've")))
print(f"I've we've you've they've:", sp.EncodeAsPieces("I've we've you've they've"))
print(f"I've we've you've they've:", sp.Encode("I've we've you've they've"))
print(f"I've we've you've they've:", sp.Decode(sp.Encode("I've we've you've they've")))

[['<unk>', 0], ['<s>', 1], ['</s>', 2], ['<0x00>', 3], ['<0x01>', 4], ['<0x02>', 5], ['<0x03>', 6], ['<0x04>', 7], ['<0x05>', 8], ['<0x06>', 9], ['<0x07>', 10], ['<0x08>', 11], ['<0x09>', 12], ['<0x0A>', 13], ['<0x0B>', 14], ['<0x0C>', 15], ['<0x0D>', 16], ['<0x0E>', 17], ['<0x0F>', 18], ['<0x10>', 19], ['<0x11>', 20], ['<0x12>', 21], ['<0x13>', 22], ['<0x14>', 23], ['<0x15>', 24], ['<0x16>', 25], ['<0x17>', 26], ['<0x18>', 27], ['<0x19>', 28], ['<0x1A>', 29], ['<0x1B>', 30], ['<0x1C>', 31], ['<0x1D>', 32], ['<0x1E>', 33], ['<0x1F>', 34], ['<0x20>', 35], ['<0x21>', 36], ['<0x22>', 37], ['<0x23>', 38], ['<0x24>', 39], ['<0x25>', 40], ['<0x26>', 41], ['<0x27>', 42], ['<0x28>', 43], ['<0x29>', 44], ['<0x2A>', 45], ['<0x2B>', 46], ['<0x2C>', 47], ['<0x2D>', 48], ['<0x2E>', 49], ['<0x2F>', 50], ['<0x30>', 51], ['<0x31>', 52], ['<0x32>', 53], ['<0x33>', 54], ['<0x34>', 55], ['<0x35>', 56], ['<0x36>', 57], ['<0x37>', 58], ['<0x38>', 59], ['<0x39>', 60], ['<0x3A>', 61], ['<0x3B>', 62], ['<0x3C

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input_format: 
  model_prefix: output/spm_model
  model_type: BPE
  vocab_size: 288
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_noi